# Plot Averaged Layer 2/3 vs Layer 5 Decoder Errors across Sessions

This notebook loads the trial-averaged decoder error traces for all Layer 2/3 and Layer 5 sessions across both projects, averages them within each layer, and plots them with standard error of the mean (SEM) bands on the same graph.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

import flexiznam as flz
from cottage_analysis.summary_analysis import get_session_list, load_project_trial_averaged

In [ ]:
import matplotlib.font_manager as fm
# Set style
sns.set_theme(style="ticks", context="talk")
font_path = '/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'

SESSIONS_TO_EXCLUDE = {
    "PZAG22.1b_S20260220": "1000 more frames than triggers in the treadmill recording"
}

In [ ]:
# Load data from the two projects
df_ccyp = load_project_trial_averaged(
    "ccyp_l5_3d_vision", 
    session_to_exclude=SESSIONS_TO_EXCLUDE.keys(),
    cut_treadmill=False
)
df_colasa = load_project_trial_averaged(
    "colasa_3d-vision_revisions", 
    session_to_exclude=SESSIONS_TO_EXCLUDE.keys(),
    cut_treadmill=False
)

# Combine datasets
df_all = pd.concat([df_ccyp, df_colasa], ignore_index=True)

In [ ]:
# Categorize layers and filter for closed-loop condition
cut_off = 350
df_all["layer"] = np.where(df_all["nominal_depth"] <= cut_off, "Layer 2/3", "Layer 5")
df_filtered = df_all[df_all["condition"] == "closedloop"]

print(f"Loaded {len(df_filtered[df_filtered.layer == 'Layer 2/3']['session'].unique())} Layer 2/3 sessions.")
print(f"Loaded {len(df_filtered[df_filtered.layer == 'Layer 5']['session'].unique())} Layer 5 sessions.")

In [ ]:
# Aggregate traces across sessions for each layer, expected speed, and optic flow combination
averaged_rows = []

# Group by layer, expected_OF and expected_RS
for (layer, of, speed), group_df in df_filtered.groupby(["layer", "expected_OF", "expected_RS"]):
    if len(group_df) == 0:
        continue
    
    trace_cols = ["RS_stim", "RS_error", "OF_error", "depth_error"]
    row_data = {
        "layer": layer,
        "expected_OF": of,
        "expected_RS": speed,
        "n_sessions": len(group_df)
    }
    
    for col in trace_cols:
        valid_arrays = [
            arr for arr in group_df[col].values
            if isinstance(arr, (np.ndarray, list, pd.Series))
        ]
        if len(valid_arrays) == 0:
            row_data[f"{col}_mean"] = np.nan
            row_data[f"{col}_sem"] = np.nan
            continue
            
        min_len = min(len(arr) for arr in valid_arrays)
        stacked = np.stack([arr[:min_len] for arr in valid_arrays])
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            row_data[f"{col}_mean"] = np.nanmean(stacked, axis=0)
            row_data[f"{col}_sem"] = stats.sem(stacked, axis=0, nan_policy='omit')
            
    averaged_rows.append(row_data)

averaged_df = pd.DataFrame(averaged_rows)

In [ ]:
# Get sorted unique values
unique_speeds = np.sort(averaged_df['expected_RS'].dropna().unique())
unique_ofs = np.sort(averaged_df['expected_OF'].dropna().unique())[::-1]

n_rows = len(unique_ofs)
n_cols = len(unique_speeds)

error_metrics = [
    ("RS_error", "Running Speed Decoder Error"),
    ("OF_error", "Optic Flow Decoder Error"),
    ("depth_error", "Depth Decoder Error")
]

layer_colors = {
    "Layer 2/3": "#1a5276",
    "Layer 5": "#b03a2e"
}
layer_styles = {
    "Layer 2/3": "-",
    "Layer 5": "--"
}

for col_name, title in error_metrics:
    fig, axes = plt.subplots(
        nrows=n_rows, 
        ncols=n_cols, 
        figsize=(3.4 * n_cols, 2.7 * n_rows), 
        sharex=True, 
        sharey=True
    )
    
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = np.expand_dims(axes, axis=0)
    elif n_cols == 1:
        axes = np.expand_dims(axes, axis=1)
        
    shared_ax_rs = None
    
    for row_idx, of in enumerate(unique_ofs):
        for col_idx, speed in enumerate(unique_speeds):
            ax = axes[row_idx, col_idx]
            
            # Filter for this combination
            subset = averaged_df[
                (averaged_df['expected_RS'] == speed) & 
                (averaged_df['expected_OF'] == of)
            ]
            
            ax_rs = ax.twinx()
            if shared_ax_rs is None:
                shared_ax_rs = ax_rs
            else:
                ax_rs.sharey(shared_ax_rs)
                
            # Plot each layer
            for layer in ["Layer 2/3", "Layer 5"]:
                layer_subset = subset[subset["layer"] == layer]
                if len(layer_subset) == 0:
                    continue
                row = layer_subset.iloc[0]
                
                # 1. Plot median running speed trace in the background
                rs_mean = row['RS_stim_mean']
                if isinstance(rs_mean, np.ndarray):
                    rs_style = ":" if layer == "Layer 5" else "--"
                    ax_rs.plot(rs_mean, color='black', linestyle=rs_style, alpha=0.15, label=f'Mean RS ({layer})')
                
                # 2. Plot error trace with filled SEM bands
                mean_trace = row[f"{col_name}_mean"]
                sem_trace = row[f"{col_name}_sem"]
                
                if isinstance(mean_trace, np.ndarray):
                    color = layer_colors[layer]
                    style = layer_styles[layer]
                    timepoints = np.arange(len(mean_trace))
                    ax.plot(timepoints, mean_trace, color=color, linestyle=style, linewidth=2.5, label=layer)
                    ax.fill_between(
                        timepoints,
                        mean_trace - sem_trace,
                        mean_trace + sem_trace,
                        color=color,
                        alpha=0.15 if layer == "Layer 2/3" else 0.08
                    )
            
            if col_idx < n_cols - 1:
                ax_rs.yaxis.set_ticklabels([])
            else:
                if row_idx == 0:
                    ax_rs.set_ylabel('Speed (m/s)', fontsize=10)
                    
            if row_idx == 0:
                ax.set_title(f"expected RS: {speed} cm/s", fontsize=11, fontweight='bold', pad=8)
                
            if col_idx == n_cols - 1:
                ax.text(
                    1.28, 0.5, f"expected OF:\n{of}", 
                    transform=ax.transAxes, 
                    va='center', ha='left', fontsize=10,
                    fontweight='semibold'
                )
                
            ax.axhline(0, color='grey', zorder=-2, linestyle=':')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            
    # Handle legend carefully to avoid duplicate labels in the legend
    handles, labels = axes[0, 0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ordered_labels = ["Layer 2/3", "Layer 5"]
    ordered_handles = [by_label[lbl] for lbl in ordered_labels if lbl in by_label]
    axes[0, 0].legend(ordered_handles, ordered_labels, loc='upper right', frameon=False, fontsize=10)

    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    fig.text(0.5, 0.01, 'Frames', ha='center', fontsize=12)
    fig.text(0.01, 0.5, 'Decoder Error ($R^2$ or MSE)', va='center', rotation='vertical', fontsize=12)

    plt.tight_layout()
    plt.show()